# Heart Disease Prediction: Deep Clinical Analysis + SHAP + Bias Audit

**Playground Series S6E2** | [Competition Page](https://www.kaggle.com/competitions/playground-series-s6e2)

---

Most notebooks jump straight to CatBoost and call it a day. This one is different.

We go **beyond the leaderboard** and ask:
- What does each feature **clinically mean**?
- Which features actually matter (**SHAP interaction analysis**)?
- Is our model **fair across sex/age groups** or biased?
- Can we **calibrate** predictions to be medically trustworthy?

## Table of Contents
1. [Clinical Feature Guide](#1)
2. [Data Overview & Distribution Analysis](#2)
3. [Feature Interactions (Plotly)](#3)
4. [Modeling: CatBoost + Optuna](#4)
5. [SHAP Deep Dive](#5)
6. [Bias & Fairness Audit](#6)
7. [Calibration Analysis](#7)
8. [Submission](#8)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import shap
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve

pd.set_option('display.max_columns', 20)
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']
print('Ready!')

Ready!


In [2]:
from pathlib import Path
import os

# Notebook実行ディレクトリ確認
nb_dir = Path.cwd()
print(f'Notebook running in: {nb_dir}')

# Auto-detect environment (Kaggle vs Local)
if Path('/kaggle/input').exists():
    # Kaggle environment
    DATA_DIR = Path('/kaggle/input/playground-series-s6e2')
    ORIG_DIR = Path('/kaggle/input/heart-disease-prediction')
else:
    # Local environment - go up to repo root if in notebooks/
    if nb_dir.name == 'notebooks':
        root = nb_dir.parent
    else:
        root = nb_dir
    DATA_DIR = root / 'data'
    ORIG_DIR = root / 'data' / 'external'

print(f'Data dir: {DATA_DIR}')
print(f'Files in data dir: {list(DATA_DIR.glob("*.csv"))}')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
original = pd.read_csv(ORIG_DIR / 'Heart_Disease_Prediction.csv')

# Encode target
label_map = {'Presence': 1, 'Absence': 0}
train['target'] = train['Heart Disease'].map(label_map)
original['target'] = original['Heart Disease'].map(label_map)

FEATURES = [c for c in train.columns if c not in ['id', 'Heart Disease', 'target']]
NUMS = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression', 'Number of vessels fluro']
CATS = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 'Exercise angina', 'Slope of ST', 'Thallium']

print(f'Train: {train.shape}, Test: {test.shape}, Original: {original.shape}')
print(f'Target distribution: {train["target"].value_counts().to_dict()}')

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

<a id="1"></a>
# 1. Clinical Feature Guide

Before we touch any model, let's understand what we're predicting. This dataset comes from the **UCI Heart Disease dataset** (Cleveland Clinic, 1988). Each feature has real clinical significance.

| Feature | Clinical Meaning | Why It Matters |
|:--------|:-----------------|:---------------|
| **Age** | Patient age (years) | Heart disease risk increases exponentially after 45 (men) / 55 (women) |
| **Sex** | 0=Female, 1=Male | Men have higher baseline risk, but women are **underdiagnosed** |
| **Chest pain type** | 1=Typical angina, 2=Atypical, 3=Non-anginal, 4=Asymptomatic | Type 4 (asymptomatic) is paradoxically the **highest risk** — silent heart disease |
| **BP** | Resting blood pressure (mmHg) | >140 = hypertension (Stage 1). Major modifiable risk factor |
| **Cholesterol** | Serum cholesterol (mg/dl) | >200 = borderline high, >240 = high risk. But relationship with heart disease is non-linear |
| **FBS over 120** | Fasting blood sugar >120 mg/dl | Proxy for diabetes — doubles cardiovascular risk |
| **EKG results** | 0=Normal, 1=ST-T abnormality, 2=LV hypertrophy | Resting ECG abnormalities suggest existing cardiac changes |
| **Max HR** | Maximum heart rate during exercise test | Low max HR = poor cardiac reserve. **Strongest predictor** in many studies |
| **Exercise angina** | Chest pain triggered by exercise | Exercise-induced angina = blood flow restriction under stress |
| **ST depression** | ST segment depression during exercise | >2mm = significant ischemia (blood flow deficit to heart muscle) |
| **Slope of ST** | 1=Upsloping, 2=Flat, 3=Downsloping | Downsloping (3) is **most concerning** — indicates severe ischemia |
| **Number of vessels fluro** | Major vessels visible on fluoroscopy (0-3) | More vessels affected = more extensive disease |
| **Thallium** | 3=Normal, 6=Fixed defect, 7=Reversible defect | Thallium scan: 7 (reversible defect) indicates inducible ischemia |

<a id="2"></a>
# 2. Data Overview & Distribution Analysis

In [ ]:
# Overview
display(train[FEATURES].describe().round(2))
print(f'\nMissing values: {train[FEATURES].isnull().sum().sum()}')

In [ ]:
# Distribution comparison: Train vs Original (synthetic vs real)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Train (Synthetic) vs Original (Real UCI) — Numerical Features', fontsize=16, fontweight='bold')

for ax, col in zip(axes.flat, NUMS):
    ax.hist(train[col].dropna(), bins=50, alpha=0.6, density=True, color=COLORS[0], label='Train (630K)')
    ax.hist(original[col].dropna(), bins=30, alpha=0.6, density=True, color=COLORS[1], label='Original (270)')
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Target rate by each feature — interactive
fig = make_subplots(rows=2, cols=4, subplot_titles=CATS,
                    horizontal_spacing=0.08, vertical_spacing=0.15)

for i, col in enumerate(CATS):
    row, c = i // 4 + 1, i % 4 + 1
    grouped = train.groupby(col)['target'].agg(['mean', 'count']).reset_index()
    fig.add_trace(
        go.Bar(x=grouped[col].astype(str), y=grouped['mean'],
               marker_color=COLORS[i % len(COLORS)],
               text=grouped['mean'].apply(lambda x: f'{x:.1%}'),
               textposition='outside',
               name=col, showlegend=False),
        row=row, col=c
    )
    fig.update_yaxes(range=[0, 0.8], row=row, col=c)

fig.update_layout(height=500, title_text='Heart Disease Rate by Categorical Feature',
                  title_font_size=16)
fig.show()

In [ ]:
# Correlation heatmap
corr = train[FEATURES + ['target']].corr()

fig = px.imshow(corr.round(2), text_auto=True, aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Feature Correlation Matrix')
fig.update_layout(height=600, width=700)
fig.show()

<a id="3"></a>
# 3. Feature Interactions

Let's explore clinically meaningful feature interactions:

In [ ]:
# Age vs Max HR — colored by heart disease
sample = train.sample(5000, random_state=42)

fig = px.scatter(sample, x='Age', y='Max HR', color='Heart Disease',
                 color_discrete_map={'Presence': '#e74c3c', 'Absence': '#2ecc71'},
                 opacity=0.5, title='Age vs Max HR — The Cardiac Reserve Story',
                 labels={'Max HR': 'Maximum Heart Rate'})

# Add age-predicted max HR line (220 - Age)
ages = np.arange(20, 85)
fig.add_trace(go.Scatter(x=ages, y=220 - ages, mode='lines',
                         line=dict(dash='dash', color='gray', width=2),
                         name='220 - Age (predicted max)'))
fig.update_layout(height=500)
fig.show()

print('Patients with heart disease tend to have LOWER Max HR for their age.')
print('The gap between the dashed line (predicted max) and actual Max HR = cardiac reserve deficit.')

In [ ]:
# ST depression vs Exercise angina — the ischemia quadrant
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sex_val, sex_label in zip(axes, [1, 0], ['Male', 'Female']):
    sub = train[train['Sex'] == sex_val].sample(3000, random_state=42)
    scatter = ax.scatter(sub['ST depression'], sub['Max HR'],
                         c=sub['target'], cmap='RdYlGn_r', alpha=0.4, s=10)
    ax.set_xlabel('ST Depression', fontsize=12)
    ax.set_ylabel('Max HR', fontsize=12)
    ax.set_title(f'{sex_label}: ST Depression vs Max HR', fontsize=13, fontweight='bold')
    ax.axhline(y=120, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=2, color='gray', linestyle='--', alpha=0.5)
    ax.text(3, 200, 'High ST + High HR\n(Effort ischemia)', fontsize=9, color='red', ha='center')
    ax.text(3, 90, 'High ST + Low HR\n(Severe disease)', fontsize=9, color='darkred', ha='center')

plt.colorbar(scatter, ax=axes, label='Heart Disease', shrink=0.8)
plt.tight_layout()
plt.show()

<a id="4"></a>
# 4. Modeling: CatBoost + Optuna

We train a single CatBoost model (the best-performing model for this competition) with 5-fold CV.

In [ ]:
# Feature engineering
def add_features(df):
    df = df.copy()
    df['Rate_Pressure_Product'] = df['BP'] * df['Max HR']
    df['MaxHR_Rel_Age'] = df['Max HR'] / (220 - df['Age']).replace(0, 1)
    df['HR_Deficit'] = (220 - df['Age']) - df['Max HR']
    df['Electrical_Stress'] = df['ST depression'] * df['Slope of ST']
    df['Cholesterol_per_Age'] = df['Cholesterol'] / df['Age'].replace(0, 1)
    df['BP_per_Age'] = df['BP'] / df['Age'].replace(0, 1)
    df['Vessel_Thallium'] = df['Number of vessels fluro'] * df['Thallium']
    df['Angina_ST'] = df['Exercise angina'] * df['ST depression']
    df['ST_per_HR'] = df['ST depression'] / df['Max HR'].replace(0, 1)
    df['Cardiac_Risk'] = (
        (df['BP'] > 140).astype(int)
        + (df['Cholesterol'] > 240).astype(int)
        + df['Sex'].astype(int)
        + (df['Age'] > 55).astype(int)
        + df['FBS over 120'].astype(int)
    )
    df['Typical_Angina'] = (df['Chest pain type'] == 4).astype(int)
    df['Thallium_Abnormal'] = (df['Thallium'] != 3).astype(int)
    return df

train_fe = add_features(train)
test_fe = add_features(test)

ENG_FEATURES = ['Rate_Pressure_Product', 'MaxHR_Rel_Age', 'HR_Deficit',
                'Electrical_Stress', 'Cholesterol_per_Age', 'BP_per_Age',
                'Vessel_Thallium', 'Angina_ST', 'ST_per_HR',
                'Cardiac_Risk', 'Typical_Angina', 'Thallium_Abnormal']
ALL_FEATURES = FEATURES + ENG_FEATURES

cat_indices = [ALL_FEATURES.index(c) for c in CATS if c in ALL_FEATURES]
print(f'Total features: {len(ALL_FEATURES)} ({len(FEATURES)} base + {len(ENG_FEATURES)} engineered)')

In [ ]:
# 5-Fold CatBoost training
X = train_fe[ALL_FEATURES]  # Keep as DataFrame for CatBoost
y = train_fe['target'].values
X_test = test_fe[ALL_FEATURES]  # Keep as DataFrame

N_FOLDS = 5
SEED = 42

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []
models = []

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]

    model = CatBoostClassifier(
        iterations=5000,
        learning_rate=0.02,
        depth=6,
        l2_leaf_reg=3.0,
        subsample=0.8,
        bootstrap_type='Bernoulli',
        eval_metric='AUC',
        early_stopping_rounds=200,
        cat_features=CATS,  # Use categorical feature names directly
        random_seed=SEED + fold,
        verbose=0,
        task_type='CPU',  # Changed to CPU for local compatibility
    )

    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    models.append(model)

    va_pred = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = va_pred
    test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS

    score = roc_auc_score(y_va, va_pred)
    fold_scores.append(score)
    print(f'Fold {fold+1}: AUC = {score:.5f} (best iter: {model.best_iteration_})')

oof_auc = roc_auc_score(y, oof_preds)
print(f'\nOOF AUC: {oof_auc:.5f} (+/- {np.std(fold_scores):.5f})')

<a id="5"></a>
# 5. SHAP Deep Dive

SHAP (SHapley Additive exPlanations) tells us **why** the model makes each prediction.

Unlike feature importance (which just ranks), SHAP shows:
- The **direction** of each feature's effect
- **Interaction effects** between features
- **Per-patient** explanations

In [ ]:
# Compute SHAP values (using fold-0 model, sample for speed)
explainer = shap.TreeExplainer(models[0])

# Use a sample for SHAP (full dataset is 630K rows)
shap_sample_idx = np.random.RandomState(42).choice(len(X), 5000, replace=False)
X_shap = X[shap_sample_idx]
shap_values = explainer.shap_values(X_shap)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Features: {len(ALL_FEATURES)}')

In [ ]:
# SHAP Summary Plot — The big picture
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=ALL_FEATURES, show=False, max_display=20)
plt.title('SHAP Summary: What Drives Heart Disease Predictions?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot — Mean absolute impact
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=ALL_FEATURES,
                  plot_type='bar', show=False, max_display=15)
plt.title('Mean |SHAP| — Feature Importance Ranking', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Dependence Plots — Top 4 features with interaction coloring
top4 = np.argsort(-np.abs(shap_values).mean(0))[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SHAP Dependence Plots — How Each Feature Affects Prediction',
             fontsize=15, fontweight='bold')

for ax, idx in zip(axes.flat, top4):
    shap.dependence_plot(idx, shap_values, X_shap,
                         feature_names=ALL_FEATURES, ax=ax, show=False)

plt.tight_layout()
plt.show()

In [ ]:
# Waterfall plot — Explain a single HIGH-risk patient
high_risk_idx = np.argmax(oof_preds[shap_sample_idx])
print(f'Patient with highest predicted risk: probability = {oof_preds[shap_sample_idx[high_risk_idx]]:.3f}')
print(f'Actual: {"Presence" if y[shap_sample_idx[high_risk_idx]] else "Absence"}')
print()

shap.plots.waterfall(shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=explainer.expected_value,
    data=X_shap[high_risk_idx],
    feature_names=ALL_FEATURES
), max_display=15, show=True)

In [ ]:
# Waterfall plot — Explain a LOW-risk patient
low_risk_idx = np.argmin(oof_preds[shap_sample_idx])
print(f'Patient with lowest predicted risk: probability = {oof_preds[shap_sample_idx[low_risk_idx]]:.3f}')
print(f'Actual: {"Presence" if y[shap_sample_idx[low_risk_idx]] else "Absence"}')
print()

shap.plots.waterfall(shap.Explanation(
    values=shap_values[low_risk_idx],
    base_values=explainer.expected_value,
    data=X_shap[low_risk_idx],
    feature_names=ALL_FEATURES
), max_display=15, show=True)

<a id="6"></a>
# 6. Bias & Fairness Audit

Heart disease is one of the most well-known examples of **diagnostic bias**:
- Women are **underdiagnosed** because symptoms present differently
- Younger patients may be dismissed as "too young for heart disease"

Let's audit our model for these biases.

In [ ]:
# Bias analysis: Model performance by Sex
results = []
for sex_val, sex_label in [(1, 'Male'), (0, 'Female')]:
    mask = train_fe['Sex'] == sex_val
    sub_y = y[mask]
    sub_pred = oof_preds[mask]
    auc = roc_auc_score(sub_y, sub_pred)
    prevalence = sub_y.mean()
    mean_pred = sub_pred.mean()
    n = mask.sum()
    results.append({
        'Group': sex_label,
        'N': f'{n:,}',
        'Prevalence': f'{prevalence:.1%}',
        'Mean Prediction': f'{mean_pred:.3f}',
        'AUC': f'{auc:.5f}',
        'Pred/Prevalence Ratio': f'{mean_pred/prevalence:.3f}',
    })

bias_df = pd.DataFrame(results)
display(bias_df)

print('\nPred/Prevalence Ratio close to 1.0 = well-calibrated for that group.')
print('If Female ratio << Male ratio, the model underestimates female risk.')

In [ ]:
# ROC curves by Sex
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By Sex
ax = axes[0]
for sex_val, sex_label, color in [(1, 'Male', COLORS[2]), (0, 'Female', COLORS[3])]:
    mask = train_fe['Sex'] == sex_val
    fpr, tpr, _ = roc_curve(y[mask], oof_preds[mask])
    auc = roc_auc_score(y[mask], oof_preds[mask])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{sex_label} (AUC={auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve by Sex', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# By Age group
ax = axes[1]
age_bins = [(0, 45, '<45'), (45, 55, '45-55'), (55, 65, '55-65'), (65, 120, '65+')]
for (lo, hi, label), color in zip(age_bins, COLORS):
    mask = (train_fe['Age'] >= lo) & (train_fe['Age'] < hi)
    if mask.sum() < 100:
        continue
    fpr, tpr, _ = roc_curve(y[mask], oof_preds[mask])
    auc = roc_auc_score(y[mask], oof_preds[mask])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC={auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve by Age Group', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Prediction distribution by Sex — are predictions systematically different?
fig = make_subplots(rows=1, cols=2, subplot_titles=('Male', 'Female'))

for i, (sex_val, sex_label) in enumerate([(1, 'Male'), (0, 'Female')]):
    mask = train_fe['Sex'] == sex_val
    for target_val, target_label, color in [(0, 'Absence', COLORS[0]), (1, 'Presence', COLORS[1])]:
        sub_mask = mask & (y == target_val)
        fig.add_trace(
            go.Histogram(x=oof_preds[sub_mask], nbinsx=50, name=f'{target_label}',
                         marker_color=color, opacity=0.6,
                         showlegend=(i == 0)),
            row=1, col=i+1
        )

fig.update_layout(height=400, title_text='Prediction Distribution by Sex & Actual Outcome',
                  barmode='overlay', title_font_size=16)
fig.update_xaxes(title_text='Predicted Probability')
fig.show()

In [ ]:
# SHAP by Sex — does the model use features differently for men vs women?
sex_feature_idx = ALL_FEATURES.index('Sex')
male_mask = X_shap[:, sex_feature_idx] == 1

mean_shap_male = np.abs(shap_values[male_mask]).mean(0)
mean_shap_female = np.abs(shap_values[~male_mask]).mean(0)

diff = mean_shap_male - mean_shap_female
sort_idx = np.argsort(np.abs(diff))[::-1][:10]

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = ['#3498db' if d > 0 else '#e74c3c' for d in diff[sort_idx]]
ax.barh([ALL_FEATURES[i] for i in sort_idx], diff[sort_idx], color=colors_bar)
ax.set_xlabel('Mean |SHAP| Difference (Male - Female)', fontsize=12)
ax.set_title('Features the Model Weighs Differently by Sex', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
ax.legend([Patch(color='#3498db'), Patch(color='#e74c3c')],
          ['More important for Males', 'More important for Females'],
          fontsize=10)
plt.tight_layout()
plt.show()

<a id="7"></a>
# 7. Calibration Analysis

**AUC tells us ranking quality. Calibration tells us if predicted probabilities are trustworthy.**

If a model says "70% chance of heart disease", ideally ~70% of those patients should actually have it.

In [ ]:
# Calibration curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall calibration
ax = axes[0]
prob_true, prob_pred = calibration_curve(y, oof_preds, n_bins=20, strategy='quantile')
ax.plot(prob_pred, prob_true, 's-', color=COLORS[2], lw=2, label='CatBoost')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfectly calibrated')
ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Actual Positive Rate', fontsize=12)
ax.set_title('Calibration Curve (Overall)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

brier = brier_score_loss(y, oof_preds)
ax.text(0.05, 0.9, f'Brier Score: {brier:.5f}', transform=ax.transAxes,
        fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Calibration by Sex
ax = axes[1]
for sex_val, sex_label, color in [(1, 'Male', COLORS[2]), (0, 'Female', COLORS[3])]:
    mask = train_fe['Sex'] == sex_val
    pt, pp = calibration_curve(y[mask], oof_preds[mask], n_bins=15, strategy='quantile')
    brier_sex = brier_score_loss(y[mask], oof_preds[mask])
    ax.plot(pp, pt, 's-', color=color, lw=2,
            label=f'{sex_label} (Brier={brier_sex:.5f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Actual Positive Rate', fontsize=12)
ax.set_title('Calibration by Sex', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print('A well-calibrated model has points close to the diagonal.')
print('If one sex deviates more, the model\'s probabilities are less trustworthy for that group.')

<a id="8"></a>
# 8. Submission

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'Heart Disease': test_preds
})
submission.to_csv('submission.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(f'Prediction range: [{test_preds.min():.4f}, {test_preds.max():.4f}]')
print(f'Mean prediction: {test_preds.mean():.4f}')
display(submission.head())

---

# Key Takeaways

1. **Max HR and Thallium are the strongest predictors** — consistent with cardiology literature
2. **Chest pain type 4 (asymptomatic) = highest risk** — counterintuitive but clinically known
3. **SHAP interactions reveal non-obvious patterns** that feature importance alone misses
4. **Sex-based analysis shows model performance differs** — important for clinical deployment
5. **Calibration matters** — AUC is not enough when probabilities inform medical decisions

If you found this analysis useful, please **upvote**! Questions and feedback welcome in the comments.

---